# LatAm LC Spread PCA Decomposition
PCA on local currency sovereign spreads vs USTs — country dislocation diagnostics

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

## Cell 1 — Parameters

In [ ]:
TENOR = '10Y'

COUNTRIES = {
    'Peru':     df_perugb_cmt,
    'Mexico':   df_mbono_cmt,
    'Colombia': df_coltes_cmt,
    'Chile':    df_btpcl_cmt,
    'Brazil':   df_bntnf_cmt,
}

FOCUS_COUNTRY = 'Peru'

# PCA estimation & fitting window — set None to use full history
LOOKBACK_START = '2022-01-01'

# number of PCs to retain; None = auto via 90% cumulative variance
N_PCS = 3

# rolling window (bus days) for residual SD bands
SD_WINDOW = 252

# SD thresholds for residual band shading
SD_BANDS = [1.25, 1.65]

# rolling sum windows for factor returns
ROLLING_WINDOWS = [5, 10, 20]

# subset to plot
ROLL_DISPLAY = [5, 20]

# trailing mean window for PC score regime shading
TRAIL_WINDOW = 60

## Cell 2 — Build Spread Panel

In [ ]:
def build_spread_panel(countries, tenor, ust_df):
    ust = ust_df.set_index('Fecha')[tenor]
    frames = {}
    for name, df in countries.items():
        loc = df.set_index('Fecha')[tenor]
        idx = loc.index.intersection(ust.index)
        frames[name] = (loc.loc[idx] - ust.loc[idx]) * 100
    panel = pd.DataFrame(frames).sort_index()
    panel.index.name = 'Fecha'
    return panel

df_spreads = build_spread_panel(COUNTRIES, TENOR, df_ust_cmt)
print(f'Shape: {df_spreads.shape}')
print(f'Date range: {df_spreads.index[0].date()} to {df_spreads.index[-1].date()}')
df_spreads.tail(3)

## Cell 3 — PCA Engine

In [ ]:
def run_pca(df_spreads, lookback_start, n_pcs):
    data = df_spreads.copy()
    if lookback_start is not None:
        data = data.loc[lookback_start:]
    data = data.dropna()
    means = data.mean()
    dm = data - means
    cov = dm.cov().values
    eigvals, eigvecs = np.linalg.eigh(cov)
    order = np.argsort(eigvals)[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]
    var_exp = eigvals / eigvals.sum()
    cum_var = np.cumsum(var_exp)
    if n_pcs is None:
        n_pcs = int(np.searchsorted(cum_var, 0.90)) + 1
    n_pcs = min(n_pcs, len(eigvals))
    pcs = [f'PC{i+1}' for i in range(n_pcs)]
    loadings = pd.DataFrame(eigvecs[:, :n_pcs], index=data.columns, columns=pcs)
    scores = pd.DataFrame(dm.values @ eigvecs[:, :n_pcs], index=dm.index, columns=pcs)
    return {
        'means':             means,
        'loadings':          loadings,
        'scores':            scores,
        'eigenvalues':       eigvals,
        'var_explained':     var_exp,
        'cum_var_explained': cum_var,
        'n_pcs':             n_pcs,
    }

pca = run_pca(df_spreads, LOOKBACK_START, N_PCS)
print(f'n_pcs: {pca["n_pcs"]}')
print(f'Var explained: {[round(v*100,1) for v in pca["var_explained"][:pca["n_pcs"]]]}')
print(f'Cum var:        {pca["cum_var_explained"][pca["n_pcs"]-1]:.1%}')

## Cell 4 — Country Selection Diagnostic
Defines `decompose` (used here and in Cell 5), then compares all-countries vs ex-Brazil configs.

In [ ]:
# ── decompose defined here so Cell 4 can use it ──────────────────────────────
def decompose(pca, df_spreads, lookback_start=None):
    # filter to window first — never project out-of-regime data
    data = df_spreads.copy()
    if lookback_start is not None:
        data = data.loc[lookback_start:]
    data = data.reindex(columns=pca['means'].index)
    dm = data.subtract(pca['means']).dropna()
    pcs = pca['loadings'].columns.tolist()
    scores_full = pd.DataFrame(
        dm.values @ pca['loadings'].values,
        index=dm.index, columns=pcs
    )
    fitted = pd.DataFrame(
        scores_full.values @ pca['loadings'].values.T + pca['means'].values,
        index=dm.index, columns=pca['means'].index
    )
    residuals = data.loc[dm.index] - fitted
    contributions = {}
    for country in pca['means'].index:
        contrib = pd.DataFrame(index=scores_full.index, columns=pcs, dtype=float)
        for pc in pcs:
            contrib[pc] = scores_full[pc] * pca['loadings'].loc[country, pc]
        contributions[country] = contrib
    return {
        'fitted':        fitted,
        'residuals':     residuals,
        'contributions': contributions,
        'scores_full':   scores_full,
    }

# ── two configs ───────────────────────────────────────────────────────────────
countries_all      = COUNTRIES.copy()
countries_ex_bra   = {k: v for k, v in COUNTRIES.items() if k != 'Brazil'}

sp_all   = build_spread_panel(countries_all,    TENOR, df_ust_cmt)
sp_exbra = build_spread_panel(countries_ex_bra, TENOR, df_ust_cmt)

pca_all   = run_pca(sp_all,   LOOKBACK_START, N_PCS)
pca_exbra = run_pca(sp_exbra, LOOKBACK_START, N_PCS)

# ── print variance and loadings side by side ─────────────────────────────────
n = pca_all['n_pcs']
pcs_a = [f'PC{i+1}' for i in range(n)]
pcs_e = [f'PC{i+1}' for i in range(pca_exbra['n_pcs'])]

print('=== Variance Explained (%): All vs Ex-Brazil ===')
ve = pd.DataFrame({
    'All_VarExp':   (pca_all['var_explained'][:n]   * 100).round(1),
    'All_CumVar':   (pca_all['cum_var_explained'][:n] * 100).round(1),
    'ExBra_VarExp': (pca_exbra['var_explained'][:pca_exbra['n_pcs']] * 100).round(1),
    'ExBra_CumVar': (pca_exbra['cum_var_explained'][:pca_exbra['n_pcs']] * 100).round(1),
}, index=pcs_a[:max(n, pca_exbra['n_pcs'])])
print(ve.to_string())

print('\n=== Loadings — All countries ===')
print(pca_all['loadings'].round(4).to_string())
print('\n=== Loadings — Ex-Brazil ===')
print(pca_exbra['loadings'].round(4).to_string())

# ── Peru residual comparison ──────────────────────────────────────────────────
dcomp_all   = decompose(pca_all,   sp_all,   LOOKBACK_START)
dcomp_exbra = decompose(pca_exbra, sp_exbra, LOOKBACK_START)

def resid_stats(res_series, label):
    r = res_series.dropna()
    return {
        'Config':   label,
        'Std':      round(r.std(), 2),
        'AC(1)':    round(r.autocorr(1), 3),
        'AC(5)':    round(r.autocorr(5), 3),
        'Min':      round(r.min(), 1),
        'Max':      round(r.max(), 1),
    }

focus = FOCUS_COUNTRY if FOCUS_COUNTRY in dcomp_all['residuals'].columns else list(countries_all.keys())[0]
stats_all   = resid_stats(dcomp_all['residuals'][focus],   'All')
stats_exbra = resid_stats(dcomp_exbra['residuals'][focus], 'Ex-Brazil')
print(f'\n=== {focus} Residual Properties ===')
print(pd.DataFrame([stats_all, stats_exbra]).set_index('Config').to_string())

# ── loadings bar charts ───────────────────────────────────────────────────────
pc_colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

def _plot_loadings_bars(ax, loadings_df, title):
    ctrs = loadings_df.index.tolist()
    pcs  = loadings_df.columns.tolist()
    x    = np.arange(len(ctrs))
    w    = 0.8 / len(pcs)
    for k, pc in enumerate(pcs):
        offset = (k - (len(pcs) - 1) / 2) * w
        bars = ax.bar(x + offset, loadings_df[pc], w, label=pc,
                      color=pc_colors[k % len(pc_colors)], alpha=0.85)
        for bar in bars:
            h = bar.get_height()
            va = 'bottom' if h >= 0 else 'top'
            ax.text(bar.get_x() + bar.get_width() / 2, h, f'{h:.2f}',
                    ha='center', va=va, fontsize=7)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(ctrs, fontsize=9)
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'PCA Loadings Comparison — {TENOR}', fontsize=12, fontweight='bold')
_plot_loadings_bars(ax1, pca_all['loadings'],   'All Countries')
_plot_loadings_bars(ax2, pca_exbra['loadings'], 'Ex-Brazil')
plt.tight_layout()
plt.show()

## Cell 5 — Decompose
`decompose` defined in Cell 4. Call here on the chosen configuration.

In [ ]:
decomp = decompose(pca, df_spreads, LOOKBACK_START)
print(f'Fitted range:   {decomp["fitted"].index[0].date()} to {decomp["fitted"].index[-1].date()}')
print(f'Residual shape: {decomp["residuals"].shape}')

## Cell 6 — Factor Returns and Rolling Sums

In [ ]:
factor_returns = decomp['scores_full'].diff()
roll_factor = {w: factor_returns.rolling(w).sum() for w in ROLLING_WINDOWS}
print(f'Factor returns computed. Rolling windows: {ROLLING_WINDOWS}')

## Cell 7 — Variance Explained (Table + Bar Chart)

In [ ]:
n = pca['n_pcs']
pcs = [f'PC{i+1}' for i in range(n)]
var_df = pd.DataFrame({
    'Var Explained (%)':     (pca['var_explained'][:n] * 100).round(1),
    'Cum Var Explained (%)': (pca['cum_var_explained'][:n] * 100).round(1),
}, index=pcs)
print('=== Variance Explained ===')
print(var_df.to_string())

fig, ax = plt.subplots(figsize=(6, 4))
ve_pct = pca['var_explained'][:n] * 100
bars = ax.bar(pcs, ve_pct, color='#4C72B0', alpha=0.85, edgecolor='white')
for bar, v in zip(bars, ve_pct):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'{v:.1f}%', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Variance Explained (%)')
ax.set_title(f'Variance Explained by PC ({TENOR})')
ax.set_ylim(0, ve_pct.max() * 1.15)
plt.tight_layout()
plt.show()

print('\n=== Loadings (country x PC) ===')
print(pca['loadings'].round(4).to_string())

## Cell 8 — Loadings Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
_plot_loadings_bars(ax, pca['loadings'], f'PCA Loadings by Country ({TENOR})')
plt.tight_layout()
plt.show()

## Cell 9 — Main Diagnostic Chart (FOCUS_COUNTRY)

In [ ]:
# ── shared band-shading helper ────────────────────────────────────────────────
def add_residual_bands(ax, resid, sd_window, sd_bands):
    roll_sd = resid.rolling(sd_window, min_periods=sd_window // 2).std()
    inner, outer = sd_bands[0], sd_bands[1]
    idx = resid.index
    for i in range(len(idx) - 1):
        sd_i = roll_sd.iloc[i]
        if pd.isna(sd_i) or sd_i == 0:
            continue
        z = resid.iloc[i] / sd_i
        x0, x1 = idx[i], idx[i + 1]
        if abs(z) > outer:
            col = 'red' if z > 0 else 'dodgerblue'
            ax.axvspan(x0, x1, color=col, alpha=0.30, linewidth=0)
        elif abs(z) > inner:
            col = 'lightsalmon' if z > 0 else 'lightblue'
            ax.axvspan(x0, x1, color=col, alpha=0.40, linewidth=0)
    for thresh in sd_bands:
        ax.plot(idx, roll_sd * thresh,  color='tomato',    lw=0.9, ls='--', alpha=0.8)
        ax.plot(idx, -roll_sd * thresh, color='steelblue', lw=0.9, ls='--', alpha=0.8)
    ax.axhline(0, color='black', lw=0.8)
    ax.plot(idx, resid, color='black', lw=1.2)
    return roll_sd

# ── pull data for focus country ────────────────────────────────────────────────
c       = FOCUS_COUNTRY
actual  = df_spreads.loc[decomp['fitted'].index, c]
fitted  = decomp['fitted'][c]
resid   = decomp['residuals'][c]
contrib = decomp['contributions'][c]
means_c = pca['means'][c]
pcs_list = pca['loadings'].columns.tolist()

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True,
                          gridspec_kw={'height_ratios': [2, 1]})
fig.suptitle(
    f'{c} | {TENOR} Spread vs UST | PCA from {LOOKBACK_START or "full history"}',
    fontsize=13, fontweight='bold'
)

# ── top panel: stacked contributions + actual ─────────────────────────────────
ax  = axes[0]
idx = contrib.index
base    = pd.Series(means_c, index=idx)  # mean as base level
pos_top = base.copy()
neg_bot = base.copy()
for k, pc in enumerate(pcs_list):
    vals = contrib[pc]
    pos  = vals.clip(lower=0)
    neg  = vals.clip(upper=0)
    ax.fill_between(idx, pos_top, pos_top + pos,
                    color=pc_colors[k % len(pc_colors)], alpha=0.65, label=pc)
    ax.fill_between(idx, neg_bot + neg, neg_bot,
                    color=pc_colors[k % len(pc_colors)], alpha=0.65)
    pos_top = pos_top + pos
    neg_bot = neg_bot + neg
ax.axhline(means_c, color='gray', lw=0.9, ls=':', label=f'Mean ({means_c:.0f} bps)')
ax.plot(idx, actual.loc[idx], color='black', lw=2.0, label='Actual', zorder=5)
ax.set_ylabel('Spread (bps)')
ax.legend(loc='upper left', fontsize=8, ncol=len(pcs_list) + 2)

# ── bottom panel: residual with bands ─────────────────────────────────────────
add_residual_bands(axes[1], resid, SD_WINDOW, SD_BANDS)
axes[1].set_ylabel('Residual (bps)')
axes[1].set_xlabel('Date')
patches = [
    mpatches.Patch(color='lightsalmon', alpha=0.6, label=f'>{SD_BANDS[0]}σ pos'),
    mpatches.Patch(color='red',         alpha=0.4, label=f'>{SD_BANDS[1]}σ pos'),
    mpatches.Patch(color='lightblue',   alpha=0.6, label=f'>{SD_BANDS[0]}σ neg'),
    mpatches.Patch(color='dodgerblue',  alpha=0.4, label=f'>{SD_BANDS[1]}σ neg'),
]
axes[1].legend(handles=patches, loc='upper left', fontsize=7, ncol=2)
plt.tight_layout()
plt.show()

## Cell 10 — All Countries Residual Dashboard

In [ ]:
ctrs  = list(decomp['residuals'].columns)
ncols = min(3, len(ctrs))
nrows = (len(ctrs) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows))
axes = np.array(axes).flatten()
fig.suptitle(f'All Countries — {TENOR} Spread Residuals vs UST', fontsize=13, fontweight='bold')
for i, c in enumerate(ctrs):
    res = decomp['residuals'][c].dropna()
    add_residual_bands(axes[i], res, SD_WINDOW, SD_BANDS)
    axes[i].set_title(c, fontsize=10, fontweight='bold')
    axes[i].set_ylabel('Residual (bps)', fontsize=8)
    axes[i].tick_params(axis='x', labelsize=7)
for j in range(len(ctrs), len(axes)):
    axes[j].set_visible(False)
plt.tight_layout()
plt.show()

## Cell 11 — PC Scores in Levels with Regime Shading

In [ ]:
scores_full = decomp['scores_full']
pcs_list    = scores_full.columns.tolist()
fig, axes = plt.subplots(len(pcs_list), 1, figsize=(14, 4 * len(pcs_list)), sharex=True)
if len(pcs_list) == 1:
    axes = [axes]
fig.suptitle(f'PC Scores — Levels and Regime ({TENOR})', fontsize=13, fontweight='bold')
for i, pc in enumerate(pcs_list):
    ax = axes[i]
    s  = scores_full[pc].dropna()
    tr = s.rolling(TRAIL_WINDOW, min_periods=TRAIL_WINDOW // 2).mean()
    idx = s.index
    for j in range(len(idx) - 1):
        if pd.isna(tr.iloc[j]):
            continue
        col = '#b8e6b8' if s.iloc[j] > tr.iloc[j] else '#f5b8b8'
        ax.axvspan(idx[j], idx[j + 1], color=col, alpha=0.4, linewidth=0)
    ax.plot(idx, s,  color='#2d5f8a',  lw=1.5, label=pc)
    ax.plot(idx, tr, color='darkorange', lw=1.2, ls='--', label=f'{TRAIL_WINDOW}d mean')
    ax.axhline(0, color='black', lw=0.7)
    ax.set_title(pc, fontsize=10)
    ax.set_ylabel('Score (bps)', fontsize=9)
    p1 = mpatches.Patch(color='#b8e6b8', alpha=0.6, label='Above trailing mean')
    p2 = mpatches.Patch(color='#f5b8b8', alpha=0.6, label='Below trailing mean')
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles=[p1, p2] + handles, fontsize=8, loc='upper left')
axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.show()

## Cell 12 — Factor Return Momentum

In [ ]:
roll_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
fig, axes = plt.subplots(len(pcs_list), 1, figsize=(14, 4 * len(pcs_list)), sharex=True)
if len(pcs_list) == 1:
    axes = [axes]
fig.suptitle(f'Factor Return Momentum — Rolling Sums ({TENOR})', fontsize=13, fontweight='bold')
for i, pc in enumerate(pcs_list):
    ax = axes[i]
    for k, w in enumerate(ROLL_DISPLAY):
        rs = roll_factor[w][pc].dropna()
        ax.plot(rs.index, rs, color=roll_colors[k % len(roll_colors)], lw=1.4, label=f'{w}d')
    ax.axhline(0, color='black', lw=0.8)
    ax.set_title(pc, fontsize=10)
    ax.set_ylabel('bps', fontsize=9)
    ax.legend(loc='upper left', fontsize=8)
axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.show()

## Cell 13 — Current Snapshot Table

In [ ]:
last = decomp['residuals'].dropna(how='all').index[-1]
pcs_list = decomp['scores_full'].columns.tolist()
rows = []
for c in decomp['residuals'].columns:
    fit = decomp['fitted'].loc[last, c]
    res = decomp['residuals'].loc[last, c]
    act = fit + res
    r_ser  = decomp['residuals'][c].dropna()
    roll_sd = r_ser.rolling(SD_WINDOW, min_periods=SD_WINDOW // 2).std()
    sd_val  = roll_sd.get(last, np.nan)
    z = res / sd_val if (not pd.isna(sd_val) and sd_val > 0) else np.nan
    row = {'Country': c, 'Actual (bps)': act, 'Fitted (bps)': fit,
           'Resid (bps)': res, 'Resid Z': z}
    for pc in pcs_list:
        row[f'{pc} contrib'] = decomp['contributions'][c].loc[last, pc]
    rows.append(row)
snap = pd.DataFrame(rows).set_index('Country')
print(f'=== Snapshot as of {last.date()} | Tenor: {TENOR} ===')
print(snap.round(1).to_string())